# UNUSED IN THE END

# School Boundary Cluster Construction
This notebook builds school-specific housing clusters around the 1km boundary. It first selects resale flats within a narrow ring around each school, then creates alternative cluster assignments for later school-level heterogeneity checks.


In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
import geopandas as gpd
from sklearn.cluster import KMeans

In [ ]:
PROCESSED_DIR = Path("../data/processed")

housing_df = pd.read_csv(PROCESSED_DIR / "final_resale_data.csv")
school_df = gpd.read_file(PROCESSED_DIR / "schools/final_primary_schools_with_school_boundaries.geojson")

# creates a geometry column for housing_df and converts to SVY21 / meters
housing_df = gpd.GeoDataFrame(
    housing_df.copy(),
    geometry=gpd.points_from_xy(housing_df["longitude"], housing_df["latitude"]),
    crs="EPSG:4326"
).to_crs("EPSG:3414")

# creates a new df where geometry is a point 
school_point_df = gpd.GeoDataFrame(
    school_df,
    geometry=gpd.points_from_xy(school_df["X"], school_df["Y"]),
    crs="EPSG:3414"
)

## Load Inputs
The setup cells load the processed resale panel and the school boundary files, then convert both housing points and school geometries into SVY21 so distance calculations are in meters.


### Set Params

In [ ]:
boundary = 100  # in meters

min_dist = 1000 - boundary
max_dist = 1000 + boundary

## Build the Boundary Ring Sample
These cells define the ring width around the 1km cutoff and collect the resale transactions that lie inside that local boundary band for each school. The resulting index lists are the inputs to the clustering step.


In [ ]:
def houses_in_ring(school_geom, housing_gdf, min_dist=900, max_dist=1100):
    dists = housing_gdf.geometry.distance(school_geom)
    mask = (dists >= min_dist) & (dists <= max_dist)
    return housing_gdf.index[mask].tolist()

school_df["house_idx_0_9_1_1km"] = school_df.geometry.apply(
    lambda geom: houses_in_ring(geom, housing_df, min_dist=min_dist, max_dist=max_dist)
)

school_df["num_houses_0_9_1_1km"] = school_df["house_idx_0_9_1_1km"].apply(len)

school_point_df["house_idx_0_9_1_1km"] = school_point_df.geometry.apply(
    lambda geom: houses_in_ring(geom, housing_df, min_dist=min_dist, max_dist=max_dist)
)

school_point_df["num_houses_0_9_1_1km"] = school_point_df["house_idx_0_9_1_1km"].apply(len)


school_df_pca = school_df.copy()


## The clustering part

## Coordinate-Only Clustering
This specification clusters nearby flats using only their projected x and y coordinates. It is the simpler geography-only partition that preserves local spatial grouping around each school.


In [ ]:

def cluster_houses_for_school(house_idx_list, housing_gdf, n_clusters=4):
    if len(house_idx_list) == 0:
        return {
            "cluster_1": [],
            "cluster_2": [],
            "cluster_3": [],
            "cluster_4": []
        }

    selected = housing_gdf.loc[house_idx_list].copy()

    # get x/y from point geometry
    coords = np.column_stack((selected.geometry.x, selected.geometry.y))

    k = min(n_clusters, len(selected))
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    selected["cluster"] = kmeans.fit_predict(coords) + 1

    result = {}
    for i in range(1, n_clusters + 1):
        if i <= k:
            result[f"cluster_{i}"] = selected.index[selected["cluster"] == i].tolist()
        else:
            result[f"cluster_{i}"] = []

    return result


cluster_results = school_df["house_idx_0_9_1_1km"].apply(
    lambda idxs: cluster_houses_for_school(idxs, housing_df, n_clusters=4)
)

cluster_df = pd.DataFrame(cluster_results.tolist(), index=school_df.index)
school_df = pd.concat([school_df, cluster_df], axis=1)

cluster_results_point = school_point_df["house_idx_0_9_1_1km"].apply(
    lambda idxs: cluster_houses_for_school(idxs, housing_df, n_clusters=4)
)
cluster_point_df = pd.DataFrame(cluster_results_point.tolist(), index=school_point_df.index)
school_point_df = pd.concat([school_point_df, cluster_point_df], axis=1)

#### Cluster by PCA

## PCA-Based Clustering
This branch augments the clustering features with housing and amenity variables, then compresses them with PCA before assigning clusters. It is a richer alternative to the coordinate-only approach.


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans


def add_pca_columns(
    df: pd.DataFrame,
    feature_cols: list[str],
    n_components: int = 2,
    scale: bool = True,
    drop_na: bool = True,
    prefix: str = "PC"
) -> tuple[pd.DataFrame, PCA, StandardScaler | None]:
    """
    Run PCA on selected columns and add principal component columns back to the dataframe.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe.
    feature_cols : list[str]
        Columns to use for PCA.
    n_components : int, default=2
        Number of principal components to create.
    scale : bool, default=True
        Whether to standardize features before PCA.
    drop_na : bool, default=True
        If True, rows with NA in feature_cols are excluded from PCA fitting.
        Their PC columns will remain NaN in the output dataframe.
    prefix : str, default="PC"
        Prefix for PCA column names.

    Returns
    -------
    df_out : pd.DataFrame
        Copy of input dataframe with PCA columns added.
    pca_model : PCA
        Fitted PCA object.
    scaler : StandardScaler or None
        Fitted scaler if scale=True, else None.
    """
    df_out = df.copy()

    missing_cols = [col for col in feature_cols if col not in df_out.columns]
    if missing_cols:
        raise ValueError(f"These feature columns are missing from df: {missing_cols}")

    if n_components < 1:
        raise ValueError("n_components must be at least 1.")

    X = df_out[feature_cols]

    if drop_na:
        valid_idx = X.dropna().index
        X_valid = X.loc[valid_idx]
    else:
        if X.isna().any().any():
            raise ValueError(
                "NA values found in feature_cols. Either set drop_na=True or clean your data first."
            )
        valid_idx = X.index
        X_valid = X

    if X_valid.shape[0] == 0:
        raise ValueError("No valid rows available for PCA after handling missing values.")

    if n_components > min(X_valid.shape[0], X_valid.shape[1]):
        raise ValueError(
            f"n_components={n_components} is too large. "
            f"Must be <= min(num_rows, num_features) = {min(X_valid.shape[0], X_valid.shape[1])}."
        )

    scaler = None
    if scale:
        scaler = StandardScaler()
        X_processed = scaler.fit_transform(X_valid)
    else:
        X_processed = X_valid.to_numpy()

    pca_model = PCA(n_components=n_components)
    X_pca = pca_model.fit_transform(X_processed)

    pc_cols = [f"{prefix}{i+1}" for i in range(n_components)]

    for col in pc_cols:
        df_out[col] = pd.NA

    df_out.loc[valid_idx, pc_cols] = X_pca

    return df_out, pca_model, scaler


def cluster_on_pcs(
    df: pd.DataFrame,
    pc_cols: list[str] | None = None,
    n_clusters: int = 4,
    cluster_col: str = "cluster",
    random_state: int = 42,
    drop_na: bool = True
) -> tuple[pd.DataFrame, KMeans]:
    """
    Cluster rows using principal component columns and add cluster labels back to the dataframe.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe containing PCA columns.
    pc_cols : list[str] or None, default=None
        PCA columns to use for clustering. If None, automatically uses columns
        starting with 'PC'.
    n_clusters : int, default=4
        Number of clusters for KMeans.
    cluster_col : str, default="cluster"
        Name of output cluster column.
    random_state : int, default=42
        Random seed for reproducibility.
    drop_na : bool, default=True
        If True, rows with NA in pc_cols are excluded from clustering.
        Their cluster labels remain NA.

    Returns
    -------
    df_out : pd.DataFrame
        Copy of input dataframe with cluster labels added.
    kmeans_model : KMeans
        Fitted KMeans model.
    """
    df_out = df.copy()

    if pc_cols is None:
        pc_cols = [col for col in df_out.columns if col.startswith("PC")]

    if not pc_cols:
        raise ValueError("No PC columns found. Pass pc_cols explicitly or run PCA first.")

    missing_cols = [col for col in pc_cols if col not in df_out.columns]
    if missing_cols:
        raise ValueError(f"These PC columns are missing from df: {missing_cols}")

    X = df_out[pc_cols]

    if drop_na:
        valid_idx = X.dropna().index
        X_valid = X.loc[valid_idx]
    else:
        if X.isna().any().any():
            raise ValueError(
                "NA values found in pc_cols. Either set drop_na=True or clean your data first."
            )
        valid_idx = X.index
        X_valid = X

    if X_valid.shape[0] == 0:
        raise ValueError("No valid rows available for clustering after handling missing values.")

    if n_clusters < 1:
        raise ValueError("n_clusters must be at least 1.")

    if n_clusters > X_valid.shape[0]:
        raise ValueError(
            f"n_clusters={n_clusters} cannot exceed number of valid rows={X_valid.shape[0]}."
        )

    kmeans_model = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    labels = kmeans_model.fit_predict(X_valid)

    df_out[cluster_col] = pd.NA
    df_out.loc[valid_idx, cluster_col] = labels

    return df_out, kmeans_model

def cluster_houses_for_school_pca(house_idx_list, housing_gdf,feature_cols, n_clusters=4):
    if len(house_idx_list) == 0:
        return {f"cluster_{i+1}": [] for i in range(n_clusters)}

    selected = housing_gdf.loc[house_idx_list].copy()

    #add PCA columns
    n_components = min(3, len(selected), len(feature_cols))
    df_pca, pca_model, scaler = add_pca_columns( selected, feature_cols=feature_cols, n_components=n_components)

    # Step 2: cluster on PCs
    df_clustered, kmeans_model = cluster_on_pcs(
        df_pca,
        pc_cols=[f"PC{i+1}" for i in range(n_components)],
        n_clusters=n_clusters,
        cluster_col="control_cluster"
    )

    k = min(n_clusters, len(selected))

    result = {}
    for i in range(n_clusters):
        if i < k:
            result[f"cluster_{i+1}"] = df_clustered.index[df_clustered["control_cluster"] == i].tolist()
        else:
            result[f"cluster_{i+1}"] = []

    return result


In [ ]:
feature_cols = [
    "year",
    "num_nearby_malls",
    "num_nearby_mrt",
    "num_unique_mrt_lines",
    "floor_area_sqm",
    "remaining_lease"
]

cluster_results_pca = school_df_pca["house_idx_0_9_1_1km"].apply(
    lambda idxs: cluster_houses_for_school_pca(idxs, housing_df,feature_cols=feature_cols, n_clusters=4)
)

cluster_df_pca = pd.DataFrame(cluster_results_pca.tolist(), index=school_df_pca.index)
school_df_pca = pd.concat([school_df_pca, cluster_df_pca], axis=1)

In [ ]:
school_df_pca

## Check that there are no weird clusters

## Visual Checks and Export
The final cells plot school-specific clusters to catch obviously implausible partitions, inspect schools with many nearby transactions, and write the cluster-enriched school files to data/processed/schools/.


In [ ]:
import matplotlib.pyplot as plt

cluster_cols = ["cluster_1", "cluster_2", "cluster_3", "cluster_4"]

cluster_colors = {
    "cluster_1": "red",
    "cluster_2": "blue",
    "cluster_3": "green",
    "cluster_4": "orange",
}

def plot_school_clusters(school_row, housing_gdf, ax):
    ax.set_aspect("equal")

    all_x, all_y = [], []

    for col in cluster_cols:
        idxs = school_row[col]

        if not isinstance(idxs, list) or len(idxs) == 0:
            continue

        pts = housing_gdf.loc[idxs]

        ax.scatter(
            pts.geometry.x,
            pts.geometry.y,
            s=5,
            c=cluster_colors[col]
        )

        all_x.extend(pts.geometry.x.tolist())
        all_y.extend(pts.geometry.y.tolist())

    # auto zoom to data
    if all_x:
        x_pad = (max(all_x) - min(all_x)) * 0.1 if max(all_x) > min(all_x) else 100
        y_pad = (max(all_y) - min(all_y)) * 0.1 if max(all_y) > min(all_y) else 100

        ax.set_xlim(min(all_x) - x_pad, max(all_x) + x_pad)
        ax.set_ylim(min(all_y) - y_pad, max(all_y) + y_pad)

    # title
    school_name = school_row.get("school_name", school_row.name)
    ax.set_title(str(school_name), fontsize=6)

    ax.set_xticks([])
    ax.set_yticks([])

def plot_school_grid(school_subset, housing_gdf, title):
    fig, axes = plt.subplots(10, 10, figsize=(24, 24))
    axes = axes.flatten()

    # turn off all first
    for ax in axes:
        ax.axis("off")

    for i, (_, row) in enumerate(school_subset.iterrows()):
        ax = axes[i]
        ax.axis("on")
        plot_school_clusters(row, housing_gdf, ax)

    fig.suptitle(title, fontsize=18)
    plt.tight_layout(rect=[0, 0, 1, 0.98])
    return fig

school_df_1 = school_df_pca.iloc[:100]
school_df_2 = school_df_pca.iloc[100:194]

fig1 = plot_school_grid(school_df_1, housing_df, "Schools 1–100")
fig2 = plot_school_grid(school_df_2, housing_df, "Schools 101–194")

plt.show()

In [ ]:
school_point_df.query("num_houses_0_9_1_1km != 0") \
    .sort_values("num_houses_0_9_1_1km", ascending=False) \
    [["school_name", "num_houses_0_9_1_1km"]]

In [ ]:
school_df.to_file(PROCESSED_DIR / "schools/final_primary_schools_w_clusters.geojson", driver="GeoJSON")
school_point_df.to_file(PROCESSED_DIR / "schools/final_primary_schools_w_clusters_point_calculation.geojson", driver="GeoJSON")
school_df_pca.to_file(PROCESSED_DIR / "schools/final_primary_schools_w_clusters_pca.geojson", driver="GeoJSON")
